# DDRI Clustering Basis Evidence

이 노트북은 `15개`와 `161개` 실험의 출발점이 된 `cluster` 컬럼의 근거를 패키지 안에서 다시 확인하기 위한 보조 근거 노트북이다.

역할은 세 가지다.

- 군집화 핵심 입력 피처를 확인
- 군집별 요약표를 다시 확인
- 대표 스테이션과 군집 가설 교차표를 다시 확인

## 1. 입력 데이터와 결과 데이터

이 노트북은 `z_final_delivery` 안에 복사된 로컬 자산만 읽는다.

- 최종 군집화 입력 피처 CSV
- 군집 요약표 CSV
- 대표 스테이션 CSV
- 군집 가설 교차표 CSV

In [1]:
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != '04_reference_docs':
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'z_final_delivery/01_analysis_ml_final/04_reference_docs'

PACKAGE_DIR = NOTEBOOK_DIR.parent
INPUT_DIR = PACKAGE_DIR / '02_input_data'

feature_df = pd.read_csv(INPUT_DIR / 'ddri_final_district_clustering_features_train_2023_2024.csv')
cluster_summary_df = pd.read_csv(INPUT_DIR / 'ddri_second_cluster_summary.csv')
rep_station_df = pd.read_csv(INPUT_DIR / 'ddri_second_cluster_representative_stations.csv')
hypothesis_df = pd.read_csv(INPUT_DIR / 'ddri_second_cluster_hypothesis_crosstab.csv')

print('feature rows:', len(feature_df))
print('cluster summary rows:', len(cluster_summary_df))
print('representative station rows:', len(rep_station_df))
print('hypothesis crosstab rows:', len(hypothesis_df))

feature rows: 164
cluster summary rows: 5
representative station rows: 23
hypothesis crosstab rows: 5


## 2. 군집화 핵심 입력 피처

핵심 입력 7개는 아래 의미를 가진다.

- `arrival_7_10_ratio`: 07~10시 반납 비율
- `arrival_11_14_ratio`: 11~14시 반납 비율
- `arrival_17_20_ratio`: 17~20시 반납 비율
- `morning_net_inflow`: 아침 순유입
- `evening_net_inflow`: 저녁 순유입
- `subway_distance_m`: 최근접 지하철 거리
- `bus_stop_count_300m`: 300m 내 버스정류장 수

즉 시간대 패턴 + 순유입 + 교통 접근성을 같이 쓴 군집화다.

In [2]:
core_cols = [
    'station_id', 'mapped_dong_code', 'mapped_dong_name',
    'arrival_7_10_ratio', 'arrival_11_14_ratio', 'arrival_17_20_ratio',
    'morning_net_inflow', 'evening_net_inflow',
    'subway_distance_m', 'bus_stop_count_300m'
]
display(feature_df[core_cols].head(10))

,station_id,mapped_dong_code,mapped_dong_name,arrival_7_10_ratio,arrival_11_14_ratio,arrival_17_20_ratio,morning_net_inflow,evening_net_inflow,subway_distance_m,bus_stop_count_300m
0,2301,11680510,신사동,0.075694,0.126693,0.373307,7.0,1754.0,676.393220,20
1,2302,11680521,논현1동,0.128852,0.165705,0.380852,-125.0,1922.0,129.301580,40
2,2303,11680521,논현1동,0.117521,0.158425,0.359829,-2241.0,910.0,31.250222,40
3,2304,11680531,논현2동,0.161560,0.183048,0.319936,-751.0,-1352.0,455.995586,12
4,2305,11680531,논현2동,0.429047,0.202975,0.167156,1661.0,-1498.0,683.890247,16
5,2306,11680545,압구정동,0.104527,0.185398,0.421350,-3991.0,3268.0,23.266638,24
6,2307,11680545,압구정동,0.094934,0.191999,0.335793,-501.0,-139.0,173.998063,20
7,2308,11680545,압구정동,0.091091,0.209470,0.331398,-1320.0,689.0,484.737255,20
8,2309,11680565,청담동,0.101803,0.185843,0.316013,-2290.0,-617.0,353.456538,32
9,2310,11680565,청담동,0.142656,0.179480,0.226001,613.0,-454.0,420.169886,28


## 3. 군집별 요약표

여기서 각 군집이 어떤 성격으로 읽혔는지 다시 확인한다.

- `cluster 0`: 업무/상업 혼합형
- `cluster 1`: 아침 도착 업무 집중형
- `cluster 2`: 주거 도착형
- `cluster 3`: 생활·상권 혼합형
- `cluster 4`: 외곽 주거형

In [3]:
display(cluster_summary_df)

,cluster,arrival_7_10_ratio,arrival_11_14_ratio,arrival_17_20_ratio,morning_net_inflow,evening_net_inflow,subway_distance_m,bus_stop_count_300m,station_count
0,0,0.2832,0.1874,0.2326,1162.9388,-792.3469,472.0807,27.1837,49
1,1,0.4863,0.1599,0.1614,10421.0000,-4702.0000,435.6681,46.6667,3
2,2,0.1134,0.1497,0.3831,-2016.6562,1214.9688,390.4809,30.9688,32
3,3,0.1532,0.1975,0.3066,-489.2131,20.6066,282.7468,27.5246,61
4,4,0.1569,0.1589,0.3123,-842.8947,477.3684,1616.5686,24.8421,19


## 4. 대표 스테이션

대표 15개 스테이션은 이 군집 결과를 바탕으로 각 군집에서 선별된 결과다.

즉 `05_prediction_long`의 대표 15개 실험은 이 표에서 이어진다.

In [4]:
display(rep_station_df)

,station_id,mapped_dong_code,mapped_dong_name,station_name,주소,latitude,longitude,total_return_count,return_7_10_count,return_11_14_count,...,life_pop_17_20_mean,arrival_7_10_ratio,arrival_11_14_ratio,arrival_17_20_ratio,morning_net_inflow,evening_net_inflow,subway_distance_m,bus_stop_count_300m,cluster,center_distance
0,4908,11680531,논현2동,SB타워 앞,강남구 도산대로 318,37.522110,127.036461,6688,1875,1282,...,48357.276452,0.280353,0.191687,0.217554,1091.0,-732.0,661.368803,32,0,0.620905
1,2328,11680640,역삼1동,르네상스 호텔 사거리 역삼지하보도 7번출구 앞,서울특별시 강남구 언주로 506,37.503212,127.042732,12169,3636,2475,...,117683.775997,0.298792,0.203386,0.217520,739.0,-443.0,497.146538,32,0,0.761847
2,4902,11680640,역삼1동,구역삼세무서 교차로,강남구 논현로 337,37.495243,127.039375,15530,4503,2797,...,117683.775997,0.289955,0.180103,0.220412,1085.0,-767.0,651.200311,20,0,0.764838
3,2364,11680545,압구정동,도산공원교차로,서울특별시 강남구 도산대로 229,37.521240,127.031898,9559,2507,1905,...,48026.726368,0.262266,0.199289,0.214562,1037.0,-1410.0,715.823222,28,0,0.863017
4,4913,11680580,삼성1동,레베쌍트 빌딩 앞,봉은사로86길 6,37.512920,127.056427,7854,2005,1528,...,46531.806533,0.255284,0.194551,0.230074,1158.0,-292.0,277.576819,20,0,0.903989
5,2377,11680750,수서동,수서역 5번출구,서울특별시 강남구 수서동 724-4,37.487350,127.102325,40303,14768,6349,...,23695.385420,0.366424,0.157532,0.220654,10143.0,-3075.0,37.046495,64,1,2.648127
6,2323,11680610,대치2동,주식회사 오뚜기 정문 앞,서울특별시 강남구 영동대로 308,37.502213,127.067207,10069,5885,1653,...,55294.004492,0.584467,0.164167,0.110835,5101.0,-3404.0,685.598306,28,1,3.268810
7,2348,11680580,삼성1동,포스코사거리(기업은행),서울특별시 강남구 테헤란로 501,37.507233,127.056854,37680,19146,5957,...,46531.806533,0.508121,0.158094,0.152654,16019.0,-7627.0,584.359557,48,1,3.345411
8,2312,11680565,청담동,청담역 13번 출구 앞,서울특별시 강남구 영동대로 703,37.520580,127.056328,6181,667,858,...,42721.223895,0.107911,0.138812,0.359489,-2109.0,465.0,295.351335,36,2,0.896887
9,2354,11680580,삼성1동,청담역 2번출구,서울특별시 강남구 영동대로 647,37.519787,127.056763,3144,369,482,...,46531.806533,0.117366,0.153308,0.375954,-325.0,473.0,304.611758,28,2,0.989848


## 5. 군집 가설 교차표

군집 레이블이 실제 가설적 지구 성격과 어떻게 대응하는지 보는 표다.

이 표는 군집 라벨이 단순 번호가 아니라, 이후 모델 해석에 쓰이는 공간적 의미를 가진다는 점을 확인하는 근거다.

In [5]:
display(hypothesis_df)

,cluster,업무/상업지구 후보,주거지구 후보
0,0,32,17
1,1,3,0
2,2,0,32
3,3,1,60
4,4,1,18


## 6. 최종 연결

이 군집 근거는 두 경로로 이어진다.

- `15개 대표 스테이션`: 군집별 대표 스테이션 선정 및 피처 탐색
- `161개 전체 스테이션`: 동일 군집 기준을 `cluster` 컬럼으로 연결해 운영 데이터 전체 비교

따라서 이 노트북은 최종 패키지에서 `cluster` 컬럼의 출처를 닫는 역할을 한다.